In [ ]:
import pandas as pd
import wrds 
import numpy as np
import tidyfinance as tf
import sqlite3

In [ ]:
# Read the CSV file
characteristics = pd.read_csv('YOUR PATH TO THE CHARACTERISTICS CSV FROM XIUS WEBSITE', low_memory=False)

# Rename 'DATE' column to 'month'
characteristics.rename(columns={'DATE': 'month'}, inplace=True)

# Convert 'month' to datetime and floor to the month (i.e., set day to the first of the month)
characteristics['month'] = pd.to_datetime(characteristics['month'], format='%Y%m%d')

characteristics['month'] = characteristics['month'].dt.to_period('M').dt.to_timestamp()

# Rename all columns except 'permno', 'month', and 'sic2' by adding a prefix 'characteristic_'
cols_to_prefix = {col: f"characteristic_{col}" for col in characteristics.columns 
                  if col not in ['permno', 'month', 'sic2']}
characteristics.rename(columns=cols_to_prefix, inplace=True)

# Drop rows with NA in 'sic2'
characteristics.dropna(subset=['sic2'], inplace=True)

# Convert 'sic2' to a categorical type
characteristics['sic2'] = characteristics['sic2'].astype('category')

# Display the first few rows to verify changes (optional)
print(characteristics.head())

In [ ]:
def rank_transform(x):
    """
    Transform a pandas Series x by ranking its values and scaling them to [-1, 1].
    Missing values are preserved.
    """
    # Compute ranks (pandas rank() returns NaN for missing values)
    rank_x = x.rank(method='average')
    
    # Compute the number of non-missing values
    non_missing = x.dropna()
    max_rank = non_missing.shape[0]
    min_rank = 1
    
    # If there are no non-missing values, return a Series of NA
    if max_rank == 0:
        return pd.Series([pd.NA] * len(x), index=x.index)
    else:
        # Scale the ranks to [-1, 1]
        return 2 * ((rank_x - min_rank) / (max_rank - min_rank) - 0.5)

# Identify columns that contain "characteristic"
cols = [col for col in characteristics.columns if "characteristic" in col]

# Group by 'month' and apply the rank_transform function across the identified columns
characteristics[cols] = characteristics.groupby('month')[cols].transform(rank_transform)


# Optional: Display the first few rows to verify the transformation
print(characteristics.head())


In [ ]:
# Identify columns that contain "characteristic"
cols = [col for col in characteristics.columns if "characteristic" in col]

# Group by 'month' and replace NA values in each characteristic column with the median (ignoring NA) for that group
characteristics[cols] = characteristics.groupby('month')[cols].transform(lambda x: x.fillna(x.median(skipna=True)))

# Replace any remaining NAs in these columns with 0
characteristics[cols] = characteristics[cols].fillna(0)

# Optional: Check the result
print(characteristics.head())

In [ ]:
db = sqlite3.connect('data.db')

tf.set_wrds_credentials()

crsp_monthly = tf.download_data(
    domain="wrds",
    dataset="crsp_monthly",
    start_date='1957-01-01',
    end_date='2016-12-31'
  )

crsp_monthly = (crsp_monthly
    .dropna(subset=["ret_excess", "mktcap", "mktcap_lag"])
 )

(crsp_monthly
   .to_sql(name="crsp_monthly", 
           con=db, 
           if_exists="replace",
           index=False)
 )

In [ ]:
df_macro_pred = tf.download_data(
  domain="macro_predictors",
  dataset="monthly",
  start_date='1957-01-01', 
  end_date='2016-12-31'
)

df_macro_pred.to_sql(
  name="macro_predictors",
  con=db, 
  if_exists="replace",
  index=False
)

In [ ]:
import pandas as pd
import numpy as np

def rank_transform(x):
    """
    Transform a pandas Series by ranking its values and scaling to [-1, 1].
    Missing values are preserved during ranking, following Gu, Kelly, and Xiu (2020).
    
    Args:
        x (pd.Series): Input series to transform (e.g., a macro predictor over time).
    
    Returns:
        pd.Series: Transformed series with values in [-1, 1], preserving NaNs initially.
    """
    # Compute ranks, preserving NaNs (method='average' for ties)
    rank_x = x.rank(method='average')
    
    # Get number of non-missing values
    non_missing_count = x.notna().sum()
    min_rank = 1
    max_rank = non_missing_count
    
    # Handle case with no non-missing values
    if max_rank == 0:
        return pd.Series(np.nan, index=x.index)
    
    # Scale ranks to [-1, 1]
    scaled = 2 * ((rank_x - min_rank) / (max_rank - min_rank) - 0.5)
    return scaled

# Read the macro_predictors table
macro_pred = pd.read_sql_query(
    "SELECT date, dp, ep, bm, ntis, tbl, tms, dfy, svar FROM macro_predictors", 
    db
)
macro_pred['month'] = pd.to_datetime(macro_pred['date'])

# Rename all columns except 'month' with the prefix "macro_"
macro_pred = macro_pred.rename(
    columns=lambda x: f"macro_{x}" if x != "month" else x
)

# Identify macro predictor columns
macro_cols = [col for col in macro_pred.columns if 'macro_' in col]

# Apply rank_transform to each macro predictor column over time
macro_pred[macro_cols] = macro_pred[macro_cols].apply(rank_transform)

# Substitute NaN values with 0 in macro predictor columns
macro_pred[macro_cols] = macro_pred[macro_cols].fillna(0)

# Optional: Display the first few rows to verify the transformation
print(macro_pred.head())

In [ ]:
import pandas as pd
from pandas.tseries.offsets import DateOffset

# Read crsp_monthly data
crsp_monthly = pd.read_sql_query(
    "SELECT date, permno, mktcap_lag, ret_excess FROM crsp_monthly", 
    db
)
crsp_monthly['month'] = pd.to_datetime(crsp_monthly['date'])

# Use macro_pred from Cell 1 (already scaled to [-1, 1] with NaNs filled as 0)
# macro_pred contains columns: ['month', 'macro_date', 'macro_dp', 'macro_ep', ...]



# ---------------------------
# 2. Shift macro predictors by one month and merge
# ---------------------------
# Create a shifted copy with month increased by one month
macro_shifted = macro_pred.copy()
macro_shifted['month'] = macro_shifted['month'] + pd.offsets.MonthBegin(1)

# Left join on 'month': start with the original 'month' column then join the shifted predictors
macro_predictors_final = pd.merge(
    macro_pred[['month']],
    macro_shifted, 
    on='month', 
    how='left'
)



# ---------------------------
# 3. Merge with Characteristics Data
# ---------------------------
# Ensure characteristics 'month' column is datetime
characteristics['month'] = pd.to_datetime(characteristics['month'])

# Merge characteristics with crsp_monthly on ['month', 'permno']
df = characteristics.merge(crsp_monthly, on=['month', 'permno'], how='inner')

# Merge with the macro predictors on 'month'
df = df.merge(macro_predictors_final, on='month', how='inner')

# Sort the DataFrame by month and permno
df.sort_values(['month', 'permno'], inplace=True)

# Drop macro_date
df = df.drop(columns=['macro_date'], axis=1)

# Create a macro_intercept column equal to 1
df['macro_intercept'] = 1

# ---------------------------
# 4. Select Final Columns
# ---------------------------
# Select permno, month, ret_excess, mktcap_lag, sic2,
# and any columns containing "macro" or "characteristic"
selected_cols = ['permno', 'month', 'ret_excess', 'mktcap_lag', 'sic2']
selected_cols += [col for col in df.columns if 'macro' in col or 'characteristic' in col]
df = df[selected_cols]



# Optional: Display the first few rows
print("\nFirst few rows of final df:")
df.head()

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Use df as the input data (equivalent to characteristics in the R code)
data = df  # Match R's head() which defaults to 6 rows

# Define columns for roles
id_columns = ['permno', 'month', 'mktcap_lag']
target_column = 'ret_excess'
feature_columns = [col for col in data.columns if col not in id_columns + [target_column] + ['sic2']]

# Function to create interaction terms between characteristics and macro variables
def create_interactions(X, feature_columns=feature_columns):
    X = pd.DataFrame(X, columns=feature_columns)
    characteristic_cols = [col for col in X.columns if 'characteristic' in col]
    macro_cols = [col for col in X.columns if 'macro' in col]
    interaction_df = pd.DataFrame()
    for char_col in characteristic_cols:
        for macro_col in macro_cols:
            interaction_name = f"{char_col}_x_{macro_col}"
            interaction_df[interaction_name] = X[char_col] * X[macro_col]
    return interaction_df.values

# One-hot encode sic2
one_hot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('sic2', one_hot, ['sic2']),
        ('interactions', FunctionTransformer(create_interactions), feature_columns),
    ],
    remainder='drop'  # Drop original columns not specified
)

# Create pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

# Fit the preprocessor
pipeline.fit(data)

# Transform the data
processed_data = pipeline.transform(data)

# Get feature names for one-hot encoded sic2
sic2_encoded_names = pipeline.named_steps['preprocessor'].named_transformers_['sic2'].get_feature_names_out(['sic2'])

# Get interaction feature names
interaction_names = []
characteristic_cols = [col for col in feature_columns if 'characteristic' in col]
macro_cols = [col for col in feature_columns if 'macro' in col]
for char_col in characteristic_cols:
    for macro_col in macro_cols:
        interaction_names.append(f"{char_col}_x_{macro_col}")

# Combine all feature names
feature_names = list(sic2_encoded_names) + interaction_names

# Create final DataFrame with id columns, target, and processed features
final_data = pd.concat([
    data[id_columns + [target_column]].reset_index(drop=True),
    pd.DataFrame(processed_data, columns=feature_names)
], axis=1)


print(final_data.head())

In [ ]:
import pandas as pd
import numpy as np
import os


# Process in chunks to manage memory
chunk_size = 100000  # Adjust if needed (50,000 for less memory, 200,000 for faster)
output_path = "data_scaled/characteristics_prepared"
os.makedirs(output_path, exist_ok=True)

# Dictionary to collect data by year
year_data = {}

print("Preprocessing and collecting data by year...")
for start in range(0, len(df), chunk_size):
    chunk = df.iloc[start:start + chunk_size]
    print(f"Processing rows {start} to {start + len(chunk)}...")
    
    # Transform the chunk
    processed_chunk = pipeline.transform(chunk)
    
    # Create chunk DataFrame
    chunk_prepared = pd.concat([
        chunk[id_columns + [target_column]].reset_index(drop=True),
        pd.DataFrame(processed_chunk, columns=feature_names)
    ], axis=1)
    
    # Add year column
    chunk_prepared['year'] = chunk_prepared['month'].dt.year
    
    # Collect chunks by year
    for year, group in chunk_prepared.groupby('year'):
        if year not in year_data:
            year_data[year] = []
        year_data[year].append(group.drop(columns=['year']))

# Save each year's data as a single Parquet file
print("Saving data by year...") 
for year, group_list in year_data.items():
    # Concatenate all chunks for this year
    year_df = pd.concat(group_list, ignore_index=True)
    year_file = f"{output_path}/year_{year}.parquet"
    year_df.to_parquet(
        year_file,
        index=False,
        engine='pyarrow'
    )
    print(f"Saved {year_file} with {len(year_df)} rows")

# Verify one output file
years = sorted(year_data.keys())
if years:
    sample_year = pd.read_parquet(f"{output_path}/year_{years[0]}.parquet")
    print("Sample of processed data:")
    print(sample_year.head())
    print(f"Sample shape: {sample_year.shape}")
    print(f"Total columns: {len(sample_year.columns)}")
else:
    print("No data was processed.")